# M05 — Window functions

[← Anterior](../M04-integracion-agregacion/04-lab-segmentacion.ipynb) · [Siguiente →](02-lab-ranking-ventana.ipynb)

`groupBy` **aplasta**: de 4 pedidos pasas a 2 filas (una por cliente) y pierdes el detalle. Una **ventana** calcula algo *usando el vecindario* (el mismo cliente, ordenado por fecha) y **deja las 4 filas**.

Eso sirve para “¿cuál es el 2.º pedido de este cliente?” o “¿cuánto lleva gastado hasta esta fecha?”.

Ejecuta las celdas **aquí**, en este mismo fichero. No lo copies a otro sitio.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m05')
print(spark.version, spark.sparkContext.master)


## Particionar la ventana no es reparticionar el fichero

`Window.partitionBy("customer_id")` quiere decir: “el ranking y la suma se reinician **en cada cliente**”. No mueve ficheros en disco (eso es `repartition` / `partitionBy` al escribir, M06–M07).

`orderBy("order_n_ts")` es el eje del tiempo: sin orden, “acumulado” no significa nada.

Al ejecutar verás 4 filas. `C1` tiene `order_n` 1 y 2; su `gmv_running` pasa de 10 a 40. `C2` vuelve a empezar en 1 (no continúa el 3). Si `gmv_running` bajara dentro del mismo cliente, el `orderBy` estaría mal.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, row_number, sum as fsum
from pyspark.sql.window import Window

hist = spark.createDataFrame([
    Row(customer_id="C1", order_id="O1", order_n_ts="2024-01-01", gmv=10.0),
    Row(customer_id="C1", order_id="O2", order_n_ts="2024-02-01", gmv=30.0),
    Row(customer_id="C2", order_id="O3", order_n_ts="2024-01-15", gmv=5.0),
    Row(customer_id="C2", order_id="O4", order_n_ts="2024-03-01", gmv=8.0),
])
# Misma window para el número de pedido y para el acumulado
w = Window.partitionBy("customer_id").orderBy("order_n_ts")
(
    hist.withColumn("order_n", row_number().over(w))  # 1, 2, 1, 2
    .withColumn("gmv_running", fsum("gmv").over(w))  # 10, 40, 5, 13
    .orderBy("customer_id", "order_n")
    .show()
)


**Siguiente:** [lab de ranking](02-lab-ranking-ventana.ipynb).
